# Repelón Risk Lens — V4 · Validación observacional IDEAM

**Módulo 3 (independiente, no modifica los notebooks 1 y 2).** Compara observaciones de la estación **REPELON - AUT** con CHIRPS V3 pentadal y GPM IMERG V07 **en el punto de la estación y exactamente en el mismo periodo**.

**Advertencia:** la coincidencia temporal y la unidad de los valores deben verificarse antes de interpretar sesgo o error. La validación es **puntual**, no valida automáticamente los 1.848 predios ni la clasificación de riesgo. El CSV original se mantiene intacto.

Ejecución: sube el CSV del IDEAM a `/content/drive/MyDrive/Agro/Precipitación_20261115.csv` (o ajusta `RUTA_IDEAM`). Después ejecuta de arriba abajo.

## 0. Configuración y conexión

In [ ]:
import os, shutil
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from google.colab import drive
import ee

PROJECT_ID = 'agro-513620'
RUTA_BASE = '/content/drive/MyDrive/Agro'
RUTA_IDEAM = f'{RUTA_BASE}/Precipitación_20261115.csv'
INICIO = pd.Timestamp('2026-09-16')
FIN = pd.Timestamp('2026-10-31')  # inclusivo; pentada 26-31 oct = 6 días
CHIRPS_ID = 'UCSB-CHC/CHIRPS/V3/PENTAD'
IMERG_ID = 'NASA/GPM_L3/IMERG_V07'

# Parámetros editables solo si el catálogo IDEAM confirma otra semántica
INTERVALO_MIN = 10
UMBRAL_COBERTURA = 0.95

drive.mount('/content/drive')
assert os.path.isfile(RUTA_IDEAM), f'No se encuentra {RUTA_IDEAM}. Sube el CSV a Drive.'
try:
    ee.Initialize(project=PROJECT_ID)
except Exception:
    ee.Authenticate()
    ee.Initialize(project=PROJECT_ID)
print('Conectado. Archivo IDEAM localizado.')

## 1. Lectura y auditoría del CSV original

In [ ]:
# CSV original: coma como separador decimal y código de estación con ceros iniciales
raw = pd.read_csv(RUTA_IDEAM, dtype=str, encoding='utf-8-sig')
requeridas = ['CodigoEstacion','CodigoSensor','FechaObservacion','ValorObservado','Latitud','Longitud','UnidadMedida']
assert set(requeridas).issubset(raw.columns), 'Columnas IDEAM inesperadas'
raw['fecha'] = pd.to_datetime(raw['FechaObservacion'], format='%Y %b %d %I:%M:%S %p', errors='coerce')
raw['mm'] = pd.to_numeric(raw['ValorObservado'].str.replace(',','.',regex=False), errors='coerce')
print('Registros originales:',len(raw))
print('Fechas ilegibles:',raw.fecha.isna().sum(),'Valores no numéricos:',raw.mm.isna().sum())
print('Estaciones:',raw.CodigoEstacion.unique())
print('Sensores:',raw.CodigoSensor.unique(),'Unidades:',raw.UnidadMedida.unique())
print('Fechas duplicadas:',raw.fecha.duplicated().sum())
print('Intervalos:',raw.fecha.min(),raw.fecha.max())
print('Valores negativos:',(raw.mm<0).sum(),'Valor máximo:',raw.mm.max())
assert raw.fecha.notna().all() and raw.mm.notna().all()
assert raw.CodigoEstacion.nunique()==1, 'El CSV contiene varias estaciones: separar antes de analizar'
assert raw.UnidadMedida.str.lower().eq('mm').all(), 'Unidad distinta de mm: revisar'
# Si existen duplicados conflictivos, detener para inspección manual.
conflictos = raw.groupby('fecha')['mm'].nunique()
print('Fechas duplicadas con valores contradictorios:',(conflictos>1).sum())
assert not (conflictos>1).any(), 'Hay duplicados contradictorios: no se elige un valor arbitrariamente'
obs = raw.drop_duplicates('fecha').sort_values('fecha').copy()
lat = float(obs.Latitud.iloc[0].replace(',','.'))
lon = float(obs.Longitud.iloc[0].replace(',','.'))
print('Coordenadas estación:',lat,lon)
print('Duplicados idénticos retirados de la copia en memoria:',len(raw)-len(obs))

## 2. Control de calidad del periodo de interés

In [ ]:
t = obs.loc[obs.fecha.between(INICIO, FIN + pd.Timedelta(days=1) - pd.Timedelta(microseconds=1))].copy()
esperados = len(pd.date_range(INICIO, FIN+pd.Timedelta(days=1),freq=f'{INTERVALO_MIN}min',inclusive='left'))
cobertura = len(t)/esperados
print('Registros únicos en periodo:',len(t),'de',esperados,'esperados; cobertura:',f'{cobertura:.2%}')
print('Suma de valores registrados (mm, si son incrementales):',round(t.mm.sum(),2))
print('Valores > 0:',int((t.mm>0).sum()))
print('Mayores valores observados:\n',t.mm.value_counts().head(12).to_string())
print('Número de registros exactamente 10.2 mm:',int(t.mm.eq(10.2).sum()))
assert cobertura >= UMBRAL_COBERTURA, 'Cobertura insuficiente para la comparación principal'
# Días con cobertura y precipitación
por_dia = t.groupby(t.fecha.dt.normalize()).agg(n=('mm','size'),suma_mm=('mm','sum'),max_mm=('mm','max'))
por_dia['cobertura'] = por_dia.n/(24*60/INTERVALO_MIN)
print('Días con cobertura < 95%:',int((por_dia.cobertura<.95).sum()))
display(por_dia.head(10))
print('ATENCIÓN: comprobar que ValorObservado es lluvia incremental por intervalo, no un acumulado de sensor.')

## 3. Comparación satelital puntual en Earth Engine

Se toma el píxel que contiene la estación; no el promedio municipal. Para GPM se suman las imágenes de 30 minutos multiplicadas por 0,5 horas. Para CHIRPS se suman las nueve pentadas que comienzan en el intervalo. La comparación no implica que un punto represente todos los predios.

In [ ]:
punto = ee.Geometry.Point([lon,lat])
ini, fin_excl = INICIO.strftime('%Y-%m-%d'),(FIN+pd.Timedelta(days=1)).strftime('%Y-%m-%d')
chirps_col = ee.ImageCollection(CHIRPS_ID).filterDate(ini,fin_excl).select('precipitation')
gpm_col = ee.ImageCollection(IMERG_ID).filterDate(ini,fin_excl).select('precipitation')
n_chirps, n_gpm = chirps_col.size().getInfo(),gpm_col.size().getInfo()
print('Imágenes CHIRPS:',n_chirps,'(esperadas 9)')
print('Imágenes GPM:',n_gpm,'(esperadas',len(pd.date_range(INICIO,FIN+pd.Timedelta(days=1),freq='30min',inclusive='left')),')')
assert n_chirps==9, 'CHIRPS: número de pentadas inesperado'
assert n_gpm >= .95*46*48, 'GPM: cobertura insuficiente'
chirps_total = chirps_col.sum().rename('chirps_mm')
gpm_total = gpm_col.sum().multiply(.5).rename('gpm_mm')
ch = chirps_total.reduceRegion(ee.Reducer.first(),punto,scale=5566,maxPixels=1e8).getInfo()['chirps_mm']
gp = gpm_total.reduceRegion(ee.Reducer.first(),punto,scale=11132,maxPixels=1e8).getInfo()['gpm_mm']
resumen = pd.DataFrame({'fuente':['IDEAM (suma condicional)','CHIRPS (píxel estación)','GPM (píxel estación)'],
                        'mm':[t.mm.sum(),ch,gp]})
display(resumen)
resumen.to_csv(f'{RUTA_BASE}/validacion_ideam_chirps_gpm_punto_2026.csv',index=False)
print('Resultados puntuales guardados. NO interpretar errores hasta confirmar la semántica IDEAM.')

## 4. Verificación de episodios y coherencia de sensor

In [ ]:
# Inspección de eventos extremos y repetición de valores sospechosos
positivos = t.loc[t.mm>0,['fecha','mm']].copy()
print('Máximos 20 registros positivos:')
display(positivos.nlargest(20,'mm'))
print('Conteo por valor (positivos):')
display(positivos.mm.value_counts().head(20))
# Guardar auditoría de calidad sin sobrescribir el archivo fuente
reporte = {'registros_originales':len(raw),'registros_unicos':len(obs),
           'duplicados_retirados':len(raw)-len(obs),'periodo_inicio':str(INICIO.date()),
           'periodo_fin':str(FIN.date()),'intervalos_esperados':esperados,
           'intervalos_observados':len(t),'cobertura':cobertura,
           'suma_observada_condicional_mm':float(t.mm.sum()),
           'lat':lat,'lon':lon,
           'pendiente':'Confirmar si ValorObservado representa precipitación incremental de cada intervalo y verificar anomalías del sensor'}
import json
with open(f'{RUTA_BASE}/auditoria_ideam_2026.json','w',encoding='utf-8') as f:
    json.dump(reporte,f,ensure_ascii=False,indent=2)
print('Auditoría guardada.')

## 5. Interpretación y siguiente paso

**Este notebook NO calibra el riesgo**. La suma IDEAM solo puede compararse científicamente si confirmamos que `ValorObservado` corresponde a lluvia incremental por registro y que el sensor no presenta fallas. Una vez confirmado, ampliar a comparaciones **diarias o por pentada** con datos faltantes, sesgo, MAE y, si hay suficientes días lluviosos, correlación. Si la serie IDEAM muestra anomalías, mantener la validación como **inconclusa** y buscar estaciones adicionales.

**Salida esperada:** `validacion_ideam_chirps_gpm_punto_2026.csv` y `auditoria_ideam_2026.json`. No se modifica la clasificación CHIRPS de los 1.848 predios.